# Day 4 — Customer Segmentation

## RFM Analysis and K-Means Clustering

This notebook segments purchasing customers according to:

- Recency: days since the latest purchase
- Frequency: number of orders
- Monetary: total revenue

The final customer segments will be saved as `gold_customer_segments`.

**Load the Silver table**

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("silver_marketing_interactions")

print(f"Total Silver rows: {silver_df.count():,}")
print(f"Unique customers: {silver_df.select('customer_id').distinct().count():,}")

display(silver_df.limit(10))

Total Silver rows: 10,000
Unique customers: 999


ad_spend,campaign_id,clicks,customer_age,customer_gender,customer_id,customer_region,device_type,discount_percent,impressions,interaction_date,marketing_channel,orders,product_category,product_id,revenue,unit_price,units_sold,website_visits,ingestion_timestamp,source_name,load_date,year,month,quarter,converted,discounted_unit_price
22.63,CMP004,0,44,Male,C0171,Balochistan,Mobile,0.0,4,2025-09-04,Display,0,Beauty,P004,0.0,3000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,9,Q3,0,3000.0
36.88,CMP003,0,20,Female,C0989,Sindh,Tablet,15.0,2,2025-09-02,Search,0,Electronics,P001,0.0,50000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,9,Q3,0,42500.0
83.97,CMP003,0,32,Male,C0387,Balochistan,Desktop,5.0,4,2025-04-07,Search,0,Beauty,P004,0.0,3000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,4,Q2,0,2850.0
84.76,CMP003,0,63,Female,C0599,Sindh,Tablet,20.0,4,2025-02-25,Search,0,Beauty,P004,0.0,3000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,2,Q1,0,2400.0
31.0,CMP002,0,24,Male,C0873,Sindh,Desktop,5.0,2,2025-06-20,Social Media,0,Accessories,P005,0.0,2500.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,6,Q2,0,2375.0
19.3,CMP003,0,50,Female,C0707,Punjab,Desktop,10.0,1,2025-08-17,Search,0,Beauty,P004,0.0,3000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,8,Q3,0,2700.0
101.31,CMP002,0,59,Female,C0034,Sindh,Mobile,10.0,8,2025-12-14,Social Media,0,Accessories,P005,0.0,2500.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,12,Q4,0,2250.0
56.01,CMP002,0,20,Female,C0721,Punjab,Mobile,15.0,5,2025-11-10,Social Media,0,Beauty,P004,0.0,3000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,11,Q4,0,2550.0
28.39,CMP001,0,23,Male,C0550,Balochistan,Desktop,25.0,3,2025-08-09,Email,0,Electronics,P001,0.0,50000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,8,Q3,0,37500.0
80.33,CMP002,0,39,Female,C0686,Sindh,Desktop,5.0,8,2025-03-13,Social Media,0,Electronics,P001,0.0,50000.0,0,0,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,3,Q1,0,47500.0


In [0]:
silver_df.printSchema()

root
 |-- ad_spend: double (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- clicks: long (nullable = true)
 |-- customer_age: long (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_region: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- impressions: long (nullable = true)
 |-- interaction_date: date (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- orders: long (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- website_visits: long (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_name: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- m

In [0]:
print(silver_df.columns)

['ad_spend', 'campaign_id', 'clicks', 'customer_age', 'customer_gender', 'customer_id', 'customer_region', 'device_type', 'discount_percent', 'impressions', 'interaction_date', 'marketing_channel', 'orders', 'product_category', 'product_id', 'revenue', 'unit_price', 'units_sold', 'website_visits', 'ingestion_timestamp', 'source_name', 'load_date', 'year', 'month', 'quarter', 'converted', 'discounted_unit_price']


**Filter purchasing customers**

## 1. Filter Purchasing Customers

Purchase-based RFM analysis excludes customers who never placed an order.
A valid purchase must have `converted = 1` and a non-null `order_id`.

In [0]:
purchases_df = silver_df.filter(
    (F.col("converted") == 1) &
    (F.col("orders") == 1)
)

print(f"Purchase rows: {purchases_df.count():,}")

print(
    "Purchasing customers:",
    purchases_df.select("customer_id").distinct().count()
)

print(
    "Total orders:",
    purchases_df.agg(F.sum("orders").alias("total_orders")).first()["total_orders"]
)

display(purchases_df.limit(10))

Purchase rows: 169
Purchasing customers: 155
Total orders: 169


ad_spend,campaign_id,clicks,customer_age,customer_gender,customer_id,customer_region,device_type,discount_percent,impressions,interaction_date,marketing_channel,orders,product_category,product_id,revenue,unit_price,units_sold,website_visits,ingestion_timestamp,source_name,load_date,year,month,quarter,converted,discounted_unit_price
57.76,CMP001,1,33,Female,C0167,Sindh,Mobile,5.0,5,2025-01-18,Email,1,Beauty,P004,5700.0,3000.0,2,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,1,Q1,1,2850.0
188.7,CMP003,1,60,Male,C0274,Sindh,Mobile,20.0,9,2025-11-12,Search,1,Accessories,P005,4000.0,2500.0,2,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,11,Q4,1,2000.0
51.77,CMP001,1,52,Female,C0224,Khyber Pakhtunkhwa,Desktop,25.0,5,2025-01-19,Email,1,Footwear,P003,12000.0,8000.0,2,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,1,Q1,1,6000.0
112.48,CMP002,1,39,Male,C0519,Islamabad,Tablet,30.0,8,2025-09-13,Social Media,1,Footwear,P003,11200.0,8000.0,2,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,9,Q3,1,5600.0
31.8,CMP001,1,31,Female,C0117,Islamabad,Mobile,15.0,3,2025-11-24,Email,1,Accessories,P005,6375.0,2500.0,3,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,11,Q4,1,2125.0
46.52,CMP003,1,39,Female,C0712,Khyber Pakhtunkhwa,Tablet,20.0,3,2025-10-24,Search,1,Clothing,P002,12000.0,5000.0,3,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,10,Q4,1,4000.0
68.48,CMP001,1,48,Female,C0075,Sindh,Mobile,30.0,8,2025-09-18,Email,1,Electronics,P001,105000.0,50000.0,3,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,9,Q3,1,35000.0
70.74,CMP001,1,20,Female,C0420,Balochistan,Desktop,10.0,6,2025-08-24,Email,1,Electronics,P001,135000.0,50000.0,3,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,8,Q3,1,45000.0
113.4,CMP002,1,40,Female,C0194,Balochistan,Tablet,20.0,9,2025-08-19,Social Media,1,Beauty,P004,2400.0,3000.0,1,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,8,Q3,1,2400.0
63.37,CMP003,1,52,Female,C0990,Sindh,Mobile,15.0,3,2025-05-29,Search,1,Electronics,P001,85000.0,50000.0,2,1,2026-08-21T11:17:28.394Z,synthetic_marketing_data,2026-08-21,2025,5,Q2,1,42500.0


**Identify the purchase-date column**

In [0]:
date_columns = [
    column_name
    for column_name, data_type in silver_df.dtypes
    if data_type in ("date", "timestamp")
    or "date" in column_name.lower()
    or "time" in column_name.lower()
]

print("Possible date columns:", date_columns)

Possible date columns: ['interaction_date', 'ingestion_timestamp', 'load_date']


In [0]:
if date_columns:
    display(
        purchases_df.select(*date_columns).limit(10)
    )
else:
    print("No possible date column was found.")

interaction_date,ingestion_timestamp,load_date
2025-01-18,2026-08-21T11:17:28.394Z,2026-08-21
2025-11-12,2026-08-21T11:17:28.394Z,2026-08-21
2025-01-19,2026-08-21T11:17:28.394Z,2026-08-21
2025-09-13,2026-08-21T11:17:28.394Z,2026-08-21
2025-11-24,2026-08-21T11:17:28.394Z,2026-08-21
2025-10-24,2026-08-21T11:17:28.394Z,2026-08-21
2025-09-18,2026-08-21T11:17:28.394Z,2026-08-21
2025-08-24,2026-08-21T11:17:28.394Z,2026-08-21
2025-08-19,2026-08-21T11:17:28.394Z,2026-08-21
2025-05-29,2026-08-21T11:17:28.394Z,2026-08-21


## 2. Define the RFM Analysis Date

The analysis date is one day after the latest purchase in the dataset. This keeps Recency consistent with the historical 2025 data.

In [0]:
purchase_date_summary = purchases_df.agg(
    F.min("interaction_date").alias("first_purchase_date"),
    F.max("interaction_date").alias("latest_purchase_date")
)

display(purchase_date_summary)

first_purchase_date,latest_purchase_date
2025-01-01,2025-12-30


In [0]:
latest_purchase_date = purchase_date_summary.first()["latest_purchase_date"]

analysis_date = spark.range(1).select(
    F.date_add(F.lit(latest_purchase_date), 1).alias("analysis_date")
).first()["analysis_date"]

print(f"Latest purchase date: {latest_purchase_date}")
print(f"RFM analysis date: {analysis_date}")

Latest purchase date: 2025-12-30
RFM analysis date: 2025-12-31


## 3. Calculate Customer-Level RFM Features

- Recency: days since the customer's latest purchase
- Frequency: total number of orders
- Monetary: total revenue generated by the customer

In [0]:
rfm_df = purchases_df.groupBy("customer_id").agg(
    F.max("interaction_date").alias("latest_purchase_date"),
    F.sum("orders").alias("frequency"),
    F.sum("revenue").alias("monetary")
).withColumn(
    "recency",
    F.datediff(F.lit(analysis_date), F.col("latest_purchase_date"))
).withColumn(
    "monetary",
    F.round(F.col("monetary"), 2)
).select(
    "customer_id",
    "latest_purchase_date",
    "recency",
    "frequency",
    "monetary"
)

print(f"Customers in RFM table: {rfm_df.count():,}")

display(
    rfm_df.orderBy(F.desc("monetary"))
)

Customers in RFM table: 155


customer_id,latest_purchase_date,recency,frequency,monetary
C0792,2025-04-27,248,2,155000.0
C0557,2025-12-09,22,1,150000.0
C0539,2025-07-20,164,1,142500.0
C0046,2025-09-10,112,1,142500.0
C0420,2025-08-24,129,1,135000.0
C0234,2025-03-13,293,1,127500.0
C0175,2025-09-06,116,1,127500.0
C0097,2025-07-05,179,1,120000.0
C0759,2025-05-29,216,2,110500.0
C0075,2025-09-18,104,1,105000.0


## 4. Validate the RFM Features

Check customer count, total orders, total revenue, missing values, and invalid values before clustering.

In [0]:
rfm_summary = rfm_df.agg(
    F.count("*").alias("purchasing_customers"),
    F.sum("frequency").alias("total_orders"),
    F.round(F.sum("monetary"), 2).alias("total_revenue"),
    F.min("recency").alias("minimum_recency"),
    F.max("recency").alias("maximum_recency"),
    F.min("frequency").alias("minimum_frequency"),
    F.max("frequency").alias("maximum_frequency")
)

display(rfm_summary)

purchasing_customers,total_orders,total_revenue,minimum_recency,maximum_recency,minimum_frequency,maximum_frequency
155,169,3827575.0,1,364,1,3


In [0]:
missing_values = rfm_df.select(
    *[
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in rfm_df.columns
    ]
)

display(missing_values)

customer_id,latest_purchase_date,recency,frequency,monetary
0,0,0,0,0


In [0]:
invalid_rfm = rfm_df.filter(
    (F.col("recency") < 0) |
    (F.col("frequency") <= 0) |
    (F.col("monetary") <= 0)
)

print(f"Invalid RFM rows: {invalid_rfm.count()}")

Invalid RFM rows: 0


## 5. Inspect RFM Distributions and Outliers

Examine the range, average, quartiles, and extreme values of Recency, Frequency, and Monetary.

In [0]:
display(
    rfm_df.select(
        "recency",
        "frequency",
        "monetary"
    ).summary()
)

summary,recency,frequency,monetary
count,155,155,155
mean,176.92903225806452,1.0903225806451613,24694.032258064515
stddev,104.05156748991978,0.3093298566273381,36441.31101384859
min,1,1,1750.0
25%,96,1,4500.0
50%,179,1,8100.0
75%,265,1,22800.0
max,364,3,155000.0


In [0]:
frequency_distribution = (
    rfm_df.groupBy("frequency")
    .count()
    .orderBy("frequency")
)

display(frequency_distribution)

frequency,count
1,142
2,12
3,1


In [0]:
for feature in ["recency", "frequency", "monetary"]:
    quartiles = rfm_df.approxQuantile(
        feature,
        [0.25, 0.50, 0.75],
        0.01
    )

    print(
        f"{feature}: "
        f"Q1 = {quartiles[0]}, "
        f"Median = {quartiles[1]}, "
        f"Q3 = {quartiles[2]}"
    )

recency: Q1 = 89.0, Median = 179.0, Q3 = 259.0
frequency: Q1 = 1.0, Median = 1.0, Q3 = 1.0
monetary: Q1 = 4500.0, Median = 8100.0, Q3 = 19200.0


In [0]:
display(
    rfm_df.orderBy(F.desc("monetary")).limit(10)
)

customer_id,latest_purchase_date,recency,frequency,monetary
C0792,2025-04-27,248,2,155000.0
C0557,2025-12-09,22,1,150000.0
C0046,2025-09-10,112,1,142500.0
C0539,2025-07-20,164,1,142500.0
C0420,2025-08-24,129,1,135000.0
C0175,2025-09-06,116,1,127500.0
C0234,2025-03-13,293,1,127500.0
C0097,2025-07-05,179,1,120000.0
C0759,2025-05-29,216,2,110500.0
C0075,2025-09-18,104,1,105000.0


In [0]:
display(
    rfm_df.orderBy(F.desc("recency")).limit(10)
)

customer_id,latest_purchase_date,recency,frequency,monetary
C0817,2025-01-01,364,1,8000.0
C0333,2025-01-02,363,1,18000.0
C0561,2025-01-06,359,1,4500.0
C0996,2025-01-11,354,1,19200.0
C0349,2025-01-14,351,1,19200.0
C0856,2025-01-17,348,1,10500.0
C0705,2025-01-18,347,1,1875.0
C0224,2025-01-19,346,1,12000.0
C0617,2025-01-19,346,1,4500.0
C0461,2025-01-20,345,1,6375.0


In [0]:
display(
    rfm_df.orderBy(
        F.desc("frequency"),
        F.desc("monetary")
    ).limit(10)
)

customer_id,latest_purchase_date,recency,frequency,monetary
C0421,2025-11-13,48,3,18350.0
C0792,2025-04-27,248,2,155000.0
C0759,2025-05-29,216,2,110500.0
C0834,2025-04-28,247,2,77550.0
C0969,2025-11-08,53,2,26000.0
C0525,2025-05-18,227,2,25300.0
C0712,2025-10-24,68,2,23200.0
C0775,2025-06-09,205,2,12000.0
C0857,2025-12-13,18,2,10875.0
C0569,2025-04-16,259,2,10100.0


## 6. Correlation Between RFM Features

Correlation measures the relationship between Recency, Frequency, and Monetary values.

In [0]:
correlation_results = [
    (
        "Recency vs Frequency",
        rfm_df.stat.corr("recency", "frequency")
    ),
    (
        "Recency vs Monetary",
        rfm_df.stat.corr("recency", "monetary")
    ),
    (
        "Frequency vs Monetary",
        rfm_df.stat.corr("frequency", "monetary")
    )
]

correlation_df = spark.createDataFrame(
    correlation_results,
    ["feature_pair", "correlation"]
).withColumn(
    "correlation",
    F.round("correlation", 4)
)

display(correlation_df)

feature_pair,correlation
Recency vs Frequency,-0.0978
Recency vs Monetary,-0.0404
Frequency vs Monetary,0.0964


**Transform skewed features**

In [0]:
rfm_transformed_df = (
    rfm_df
    .withColumn(
        "log_frequency",
        F.log1p(F.col("frequency"))
    )
    .withColumn(
        "log_monetary",
        F.log1p(F.col("monetary"))
    )
)

display(
    rfm_transformed_df.select(
        "customer_id",
        "recency",
        "frequency",
        "log_frequency",
        "monetary",
        "log_monetary"
    ).limit(10)
)

customer_id,recency,frequency,log_frequency,monetary,log_monetary
C0167,269,2,1.0986122886681096,9900.0,9.200391041122515
C0274,49,1,0.6931471805599453,4000.0,8.294299608857235
C0224,346,1,0.6931471805599453,12000.0,9.392745258631441
C0519,109,1,0.6931471805599453,11200.0,9.32375833901174
C0117,37,1,0.6931471805599453,6375.0,8.760296220470051
C0712,68,2,1.0986122886681096,23200.0,10.05195066017375
C0075,104,1,0.6931471805599453,105000.0,11.561725152903833
C0420,129,1,0.6931471805599453,135000.0,11.813037464800539
C0194,134,1,0.6931471805599453,2400.0,7.783640596221253
C0990,216,1,0.6931471805599453,85000.0,11.350418300109132


## 7. Assemble and Standardize Clustering Features

Combine Recency, transformed Frequency, and transformed Monetary into a feature vector. Standardize them so every feature has a comparable scale.

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

assembler = VectorAssembler(
    inputCols=[
        "recency",
        "log_frequency",
        "log_monetary"
    ],
    outputCol="unscaled_features"
)

assembled_df = assembler.transform(rfm_transformed_df)

display(
    assembled_df.select(
        "customer_id",
        "recency",
        "log_frequency",
        "log_monetary",
        "unscaled_features"
    ).limit(10)
)

customer_id,recency,log_frequency,log_monetary,unscaled_features
C0167,269,1.0986122886681096,9.200391041122515,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""269.0"",""1.0986122886681096"",""9.200391041122515""]}"
C0274,49,0.6931471805599453,8.294299608857235,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""49.0"",""0.6931471805599453"",""8.294299608857235""]}"
C0224,346,0.6931471805599453,9.392745258631441,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""346.0"",""0.6931471805599453"",""9.392745258631441""]}"
C0519,109,0.6931471805599453,9.32375833901174,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""109.0"",""0.6931471805599453"",""9.32375833901174""]}"
C0117,37,0.6931471805599453,8.760296220470051,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""37.0"",""0.6931471805599453"",""8.760296220470051""]}"
C0712,68,1.0986122886681096,10.05195066017375,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""68.0"",""1.0986122886681096"",""10.05195066017375""]}"
C0075,104,0.6931471805599453,11.561725152903833,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""104.0"",""0.6931471805599453"",""11.561725152903833""]}"
C0420,129,0.6931471805599453,11.813037464800539,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""129.0"",""0.6931471805599453"",""11.813037464800539""]}"
C0194,134,0.6931471805599453,7.783640596221253,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""134.0"",""0.6931471805599453"",""7.783640596221253""]}"
C0990,216,0.6931471805599453,11.350418300109132,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""216.0"",""0.6931471805599453"",""11.350418300109132""]}"


In [0]:
scaler = StandardScaler(
    inputCol="unscaled_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(assembled_df)
scaled_df = scaler_model.transform(assembled_df)

display(
    scaled_df.select(
        "customer_id",
        "recency",
        "frequency",
        "monetary",
        "unscaled_features",
        "features"
    ).limit(10)
)

customer_id,recency,frequency,monetary,unscaled_features,features
C0167,269,2,9900.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""269.0"",""1.0986122886681096"",""9.200391041122515""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8848590171489248"",""3.0550993501812918"",""-0.0818988918276915""]}"
C0274,49,1,4000.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""49.0"",""0.6931471805599453"",""8.294299608857235""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.2294772231130286"",""-0.2964383478344293"",""-0.8275542089634771""]}"
C0224,346,1,12000.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""346.0"",""0.6931471805599453"",""9.392745258631441""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.6248767012406087"",""-0.2964383478344293"",""0.07639633087454886""]}"
C0519,109,1,11200.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""109.0"",""0.6931471805599453"",""9.32375833901174""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.6528400666779505"",""-0.2964383478344293"",""0.01962450675670487""]}"
C0117,37,1,6375.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""37.0"",""0.6931471805599453"",""8.760296220470051""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.3448046544000443"",""-0.2964383478344293"",""-0.44406879144037553""]}"
C0712,68,2,23200.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""68.0"",""1.0986122886681096"",""10.05195066017375""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.0468754569085872"",""3.0550993501812918"",""0.6188802295643041""]}"
C0075,104,1,105000.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""104.0"",""0.6931471805599453"",""11.561725152903833""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.7008931630475403"",""-0.2964383478344293"",""1.8613281085445499""]}"
C0420,129,1,135000.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""129.0"",""0.6931471805599453"",""11.813037464800539""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.4606276811995911"",""-0.2964383478344293"",""2.0681420733990397""]}"
C0194,134,1,2400.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""134.0"",""0.6931471805599453"",""7.783640596221253""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.4125745848300012"",""-0.2964383478344293"",""-1.2477939270053642""]}"
C0990,216,1,85000.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""216.0"",""0.6931471805599453"",""11.350418300109132""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.3754961956312724"",""-0.2964383478344293"",""1.687436078752718""]}"


## 8. Select the Number of Clusters

Evaluate K-Means with k values from 2 to 6 using the silhouette score and cluster sizes. The final k will also consider business usefulness.

In [0]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

k_results = []

for k in range(2, 7):
    kmeans = KMeans(
        featuresCol="features",
        predictionCol="cluster",
        k=k,
        seed=42,
        maxIter=50
    )

    model = kmeans.fit(scaled_df)
    predictions = model.transform(scaled_df)

    silhouette = evaluator.evaluate(predictions)

    cluster_sizes = [
        row["count"]
        for row in predictions.groupBy("cluster")
        .count()
        .orderBy("cluster")
        .collect()
    ]

    k_results.append(
        (
            k,
            float(silhouette),
            str(cluster_sizes),
            min(cluster_sizes),
            max(cluster_sizes)
        )
    )

k_evaluation_df = spark.createDataFrame(
    k_results,
    [
        "k",
        "silhouette_score",
        "cluster_sizes",
        "smallest_cluster",
        "largest_cluster"
    ]
).withColumn(
    "silhouette_score",
    F.round("silhouette_score", 4)
)

display(k_evaluation_df.orderBy("k"))

k,silhouette_score,cluster_sizes,smallest_cluster,largest_cluster
2,0.7644,"[13, 142]",13,142
3,0.5161,"[72, 70, 13]",13,72
4,0.6448,"[13, 63, 51, 28]",13,63
5,0.6297,"[53, 13, 15, 21, 53]",13,53
6,0.5668,"[35, 21, 45, 13, 10, 31]",10,45


## Train the final K-Means model

In [0]:
final_k = 4

final_kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    k=final_k,
    seed=42,
    maxIter=50
)

final_kmeans_model = final_kmeans.fit(scaled_df)

clustered_df = final_kmeans_model.transform(scaled_df)

print(f"Final number of clusters: {final_k}")

display(
    clustered_df.groupBy("cluster")
    .count()
    .orderBy("cluster")
)

Final number of clusters: 4


cluster,count
0,13
1,63
2,51
3,28


## 9. Profile the Final Customer Clusters

Examine customer count, average Recency, average Frequency, average Monetary value, and total revenue for each cluster.

In [0]:
total_customers = clustered_df.count()

total_revenue = clustered_df.agg(
    F.sum("monetary").alias("total_revenue")
).first()["total_revenue"]

cluster_profile_df = (
    clustered_df.groupBy("cluster")
    .agg(
        F.count("*").alias("customer_count"),
        F.avg("recency").alias("avg_recency"),
        F.avg("frequency").alias("avg_frequency"),
        F.avg("monetary").alias("avg_monetary"),
        F.sum("monetary").alias("total_revenue")
    )
    .withColumn(
        "customer_percentage",
        F.round(
            F.col("customer_count") / F.lit(total_customers) * 100,
            2
        )
    )
    .withColumn(
        "revenue_percentage",
        F.round(
            F.col("total_revenue") / F.lit(total_revenue) * 100,
            2
        )
    )
    .select(
        "cluster",
        "customer_count",
        "customer_percentage",
        F.round("avg_recency", 2).alias("avg_recency"),
        F.round("avg_frequency", 2).alias("avg_frequency"),
        F.round("avg_monetary", 2).alias("avg_monetary"),
        F.round("total_revenue", 2).alias("total_revenue"),
        "revenue_percentage"
    )
    .orderBy("cluster")
)

display(cluster_profile_df)

cluster,customer_count,customer_percentage,avg_recency,avg_frequency,avg_monetary,total_revenue,revenue_percentage
0,13,8.39,149.54,2.08,38057.69,494750.0,12.93
1,63,40.65,110.19,1.0,5894.05,371325.0,9.7
2,51,32.9,287.22,1.0,10862.75,554000.0,14.47
3,28,18.06,138.93,1.0,85982.14,2407500.0,62.9


## Assign segment names and actions

In [0]:
segment_mapping = spark.createDataFrame(
    [
        (
            0,
            "Loyal Customers",
            "Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
        ),
        (
            1,
            "Recent Low-Value Customers",
            "Use welcome offers, product recommendations, and low-cost automated campaigns"
        ),
        (
            2,
            "At-Risk Customers",
            "Send re-engagement discounts, reminders, and limited-time win-back offers"
        ),
        (
            3,
            "High-Value One-Time Customers",
            "Provide VIP offers, premium recommendations, and incentives for a second purchase"
        )
    ],
    ["cluster", "segment_name", "recommended_action"]
)

final_segments_df = (
    clustered_df.join(
        segment_mapping,
        on="cluster",
        how="left"
    )
    .select(
        "customer_id",
        "latest_purchase_date",
        "recency",
        "frequency",
        "monetary",
        "cluster",
        "segment_name",
        "recommended_action"
    )
)

display(
    final_segments_df.orderBy(
        "cluster",
        F.desc("monetary")
    )
)

customer_id,latest_purchase_date,recency,frequency,monetary,cluster,segment_name,recommended_action
C0792,2025-04-27,248,2,155000.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0759,2025-05-29,216,2,110500.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0834,2025-04-28,247,2,77550.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0969,2025-11-08,53,2,26000.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0525,2025-05-18,227,2,25300.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0712,2025-10-24,68,2,23200.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0421,2025-11-13,48,3,18350.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0775,2025-06-09,205,2,12000.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0857,2025-12-13,18,2,10875.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0569,2025-04-16,259,2,10100.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"


## 10. Save Customer Segments to the Gold Layer

Save the final customer-level segmentation results as `gold_customer_segments` for reporting and Power BI.

In [0]:
final_segments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_customer_segments")

print("Successfully created: gold_customer_segments")

Successfully created: gold_customer_segments


In [0]:
gold_segments_df = spark.table("gold_customer_segments")

print(f"Gold table rows: {gold_segments_df.count()}")
print(
    "Unique customers:",
    gold_segments_df.select("customer_id").distinct().count()
)

display(gold_segments_df.limit(10))

Gold table rows: 155
Unique customers: 155


customer_id,latest_purchase_date,recency,frequency,monetary,cluster,segment_name,recommended_action
C0679,2025-03-25,281,1,4500.0,2,At-Risk Customers,"Send re-engagement discounts, reminders, and limited-time win-back offers"
C0727,2025-09-18,104,1,10000.0,1,Recent Low-Value Customers,"Use welcome offers, product recommendations, and low-cost automated campaigns"
C0461,2025-01-20,345,1,6375.0,2,At-Risk Customers,"Send re-engagement discounts, reminders, and limited-time win-back offers"
C0038,2025-08-24,129,1,3000.0,1,Recent Low-Value Customers,"Use welcome offers, product recommendations, and low-cost automated campaigns"
C0834,2025-04-28,247,2,77550.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0633,2025-11-11,50,1,10000.0,1,Recent Low-Value Customers,"Use welcome offers, product recommendations, and low-cost automated campaigns"
C0895,2025-12-26,5,1,1875.0,1,Recent Low-Value Customers,"Use welcome offers, product recommendations, and low-cost automated campaigns"
C0252,2025-10-29,63,1,1750.0,1,Recent Low-Value Customers,"Use welcome offers, product recommendations, and low-cost automated campaigns"
C0569,2025-04-16,259,2,10100.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"
C0857,2025-12-13,18,2,10875.0,0,Loyal Customers,"Offer loyalty rewards, referrals, and personalized repeat-purchase campaigns"


In [0]:
display(
    gold_segments_df.groupBy(
        "cluster",
        "segment_name"
    )
    .agg(
        F.count("*").alias("customers"),
        F.round(F.sum("monetary"), 2).alias("total_revenue")
    )
    .orderBy("cluster")
)

cluster,segment_name,customers,total_revenue
0,Loyal Customers,13,494750.0
1,Recent Low-Value Customers,63,371325.0
2,At-Risk Customers,51,554000.0
3,High-Value One-Time Customers,28,2407500.0


In [0]:
missing_segments = gold_segments_df.filter(
    F.col("segment_name").isNull() |
    F.col("recommended_action").isNull()
).count()

print(f"Rows with missing segment details: {missing_segments}")

Rows with missing segment details: 0


In [0]:
gold_validation = gold_segments_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.sum("frequency").alias("total_orders"),
    F.round(F.sum("monetary"), 2).alias("total_revenue"),
    F.countDistinct("segment_name").alias("number_of_segments")
)

display(gold_validation)

total_rows,unique_customers,total_orders,total_revenue,number_of_segments
155,155,169,3827575.0,4


## Final Segmentation Summary

The RFM and K-Means analysis segmented 155 purchasing customers into four actionable groups:

- **Loyal Customers:** 13 repeat customers generating PKR 494,750
- **Recent Low-Value Customers:** 63 customers generating PKR 371,325
- **At-Risk Customers:** 51 inactive customers generating PKR 554,000
- **High-Value One-Time Customers:** 28 customers generating PKR 2,407,500

The final model uses four clusters because it provides a good balance between silhouette quality, cluster size, and business usefulness.

The segmentation results were saved as:

`gold_customer_segments`